In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
import numpy as np
import pandas as pd

In [2]:
@dataclass
class AuditConfig:
    target_rel_margin: float = 0.10   # want CTR known within +-10% (relative)
    min_epv: int = 20                 # clicks per parameter (10 = warn floor)
    min_clicks_per_level: int = 20    # per categorical level
    unseen_mass_max: float = 0.001    # Good-Turing: tolerated unseen-level share
    min_rows_per_range_bin: int = 50  # per decile of the prediction range
    target_delta_rel: float = 0.10    # want to detect a 10% relative CTR diff
    power_group_share: float = 0.5    # size of the smaller group compared
    target_auc: float = 0.65          # AUC you need from the model
    lc_fractions: tuple = (0.05, 0.10, 0.20, 0.40, 0.70, 1.00)
    lc_rise_eps: float = 0.004        # AUC gain over last step that counts as "rising"


def _res(check, status, measured, required, reason, n_req=None):
    return {"check": check, "status": status, "measured": measured,
            "required": required, "reason": reason, "n_req": n_req}


In [3]:
rng = np.random.default_rng(11)

def clicks_from(logit, r):
    return r.binomial(1, 1 / (1 + np.exp(-logit)))

In [4]:
@dataclass
class AuditConfig:
    target_rel_margin: float = 0.10   # want CTR known within +-10% (relative)
    min_epv: int = 20                 # clicks per parameter (10 = warn floor)
    min_clicks_per_level: int = 20    # per categorical level
    unseen_mass_max: float = 0.001    # Good-Turing: tolerated unseen-level share
    min_rows_per_range_bin: int = 50  # per decile of the prediction range
    target_delta_rel: float = 0.10    # want to detect a 10% relative CTR diff
    power_group_share: float = 0.5    # size of the smaller group compared
    target_auc: float = 0.65          # AUC you need from the model
    lc_fractions: tuple = (0.05, 0.10, 0.20, 0.40, 0.70, 1.00)
    lc_rise_eps: float = 0.004        # AUC gain over last step that counts as "rising"


def _res(check, status, measured, required, reason, n_req=None):
    return {"check": check, "status": status, "measured": measured,
            "required": required, "reason": reason, "n_req": n_req}

#### Checks

**Absolute margin**

$$m_{\mathrm{abs}} = 1.96 \sqrt{\frac{p(1-p)}{n}}$$

**Relative margin**

$$m_{\mathrm{rel}} = \frac{m_{\mathrm{abs}}}{p} = 1.96 \sqrt{\frac{1-p}{n\,p}} \qquad (\pm x\%)$$

**Required sample size**

Setting $m_{\mathrm{rel}} = $ `target_rel_margin` and solving for $n$:

$$1.96 \sqrt{\frac{1-p}{n\,p}} = m_{\mathrm{target}} \;\Longrightarrow\; n = \frac{1.96^2 (1-p)}{p \cdot m_{\mathrm{target}}^2}$$

i.e. the number of samples required to hit `target_rel_margin`.

In [5]:
def check_precision(y, cfg):
    n, p = len(y), y.mean()
    if p == 0:
        return _res("precision", "FAIL", "0 clicks", "-",
                    "No clicks at all: nothing can be estimated.", None)
    
    rel_m = 1.96 * np.sqrt((1 - p) / (n * p))          # relative 95% margin
    n_req = int(np.ceil(1.96**2 * (1 - p) / (p * cfg.target_rel_margin**2)))
    
    status = "PASS" if rel_m <= cfg.target_rel_margin else "FAIL"
    reason = (f"CTR={p:.4f} known to +-{rel_m*100:.1f}% (relative). "
              + ("Good enough." if status == "PASS" else
                 f"Target +-{cfg.target_rel_margin*100:.0f}% needs ~{n_req:,} rows."))
    return _res("precision", status, f"+-{rel_m*100:.1f}%",
                f"+-{cfg.target_rel_margin*100:.0f}%", reason,
                None if status == "PASS" else n_req)

$$n_{\text{clicks}} \geq \texttt{\{min\_epv\}}\text{k} \;\Longrightarrow\; n_{\text{rows}} \geq \frac{\texttt{\{min\_epv\}}\text{k}}{p}$$

In [6]:
def count_parameters(df, numeric_cols, categorical_cols):
    return len(numeric_cols) + sum(df[c].nunique() - 1 for c in categorical_cols)


def check_epv(df, y, numeric_cols, categorical_cols, cfg):
    k = count_parameters(df, numeric_cols, categorical_cols)
    events = int(min(y.sum(), (1 - y).sum()))
    epv = events / k if k else np.inf
    p = y.mean()
    n_req = int(np.ceil(cfg.min_epv * k / p)) if p > 0 else None
    status = "PASS" if epv >= cfg.min_epv else ("WARN" if epv >= 10 else "FAIL")
    reason = (f"{events:,} clicks / {k} parameters = {epv:.1f} events per variable. "
              + ("Stable fit." if status == "PASS" else
                 f"Need >= {cfg.min_epv}: ~{n_req:,} rows at this CTR, "
                 f"or reduce parameters (bucket rare category levels)."))
    return _res("EPV", status, f"{epv:.1f}", f">={cfg.min_epv}", reason,
                None if status == "PASS" else n_req)

**Question it answers:** does every category level (each brand, each device, ...)
have enough clicks to be learned, and how likely are unseen levels in the future?

**What it does, per categorical feature:**

1. **Clicks per level** — `groupby(level).sum()` counts clicks in each level.
   A level with fewer than `min_clicks_per_level` (default 20) clicks is *thin*:
   its CTR cannot be estimated usably, the model will fit noise for it.
2. **Unseen-level risk (Good–Turing)** — `(# levels seen exactly once) / n`.
   Estimates the share of *future* rows that will carry a category value
   never seen in training. Must be tiny (default < 0.001).

**Verdict:** FAIL if any level is thin or unseen risk is too high; PASS otherwise.

**The fix it recommends, with the math:**

- Expected clicks in a level = `n · s · p` (rows × level share × CTR).
- So a level needs share `s ≥ min_clicks / (n · p)` to be viable.
  - Example: n = 30,000, p = 0.05, min 20 clicks → s ≥ 20/1500 ≈ **1.3%**.
- Two options, cheap vs expensive:
  - **Bucket** all levels with share below that cutoff into `OTHER`, or
  - **Collect** `min_clicks / (s_rarest · p)` rows to keep the rarest level
    separate (usually a huge number → usually not worth it).
- Edge case: if the cutoff share ≥ 50%, no bucketing can help — total clicks
  are too few; the real problem is dataset size (see precision/EPV checks).

**Output:** one line per feature with counts, risk, and the recommended fix;
overall status is the worst across features.

In [7]:
def check_levels(df, y, categorical_cols, cfg):
    n, p = len(df), y.mean()
    worst, msgs, n_req = "PASS", [], None
    for c in categorical_cols:
        clicks = df.groupby(c, observed=True)[y.name].sum()
        counts = df[c].value_counts()
        thin = clicks[clicks < cfg.min_clicks_per_level]
        singletons = int((counts == 1).sum())
        unseen = singletons / n                       # Good-Turing estimate
        if len(thin) or unseen > cfg.unseen_mass_max:
            worst = "FAIL"
            s_min = cfg.min_clicks_per_level / (n * p) if p > 0 else np.nan
            if not np.isfinite(s_min) or s_min >= 0.5:
                fix = (f"At this CTR even a 100%-share level cannot reach "
                       f"{cfg.min_clicks_per_level} clicks -- the binding problem "
                       f"is total clicks (see precision/EPV), not bucketing.")
            else:
                smallest_share = counts.min() / n
                n_req_c = int(np.ceil(
                    cfg.min_clicks_per_level / (smallest_share * p)))
                fix = (f"Fix: bucket levels with share <{s_min*100:.2f}% into "
                       f"OTHER (cheap), or collect ~{n_req_c:,} rows to keep the "
                       f"rarest level separate (usually not worth it).")
            msgs.append(
                f"'{c}': {len(thin)}/{len(clicks)} levels have <"
                f"{cfg.min_clicks_per_level} clicks; unseen-level risk "
                f"(Good-Turing) = {unseen:.4f}. {fix}")
        else:
            msgs.append(f"'{c}': all {len(clicks)} levels have >="
                        f"{cfg.min_clicks_per_level} clicks; unseen risk {unseen:.4f}.")
    return _res("levels", worst,
                "; ".join(f"{c}:{df[c].nunique()} lvls" for c in categorical_cols),
                f">={cfg.min_clicks_per_level} clicks/level", " | ".join(msgs), n_req)

**Question it answers:** *"Will the model have to predict in regions of a numeric feature where it never saw training data?"* Models interpolate well and extrapolate badly.

**How it works, step by step:**

1. For each numeric feature, take the range you plan to predict on:
   - from `pred_ranges` if you declared it (e.g. `{"price": (10, 200)}`),
   - otherwise default to the training data's own 0.5%–99.5% percentiles (then only *internal* gaps can be found).
2. Split that range into **10 equal bins** (`np.linspace(lo, hi, 11)` = 11 edges → 10 bins).
3. Count training rows in each bin (`np.histogram`).
4. Any bin with fewer than `cfg.min_rows_per_range_bin` rows (default 50) = an unsupported region → **FAIL**.

**Output:** one result with:
- `status` — PASS, or FAIL if any feature has thin/empty bins,
- `reason` — which feature, how many bad bins, the exact intervals (first 4 shown, `...` if more), and the fix: *collect data in the gaps, or don't predict there*.

**Example verdict it produces:**

> `'f_price': 7/10 bins of the prediction range [10.0, 200.0] have <50 training rows: [67.0, 86.0), [86.0, 105.0), ... The model would extrapolate there.`

**Catches two situations:**
- **edge gap** — training covered price 10–60, you want predictions up to 200;
- **internal hole** — training data is bimodal (e.g. only cheap and expensive items), the middle is empty.

**Limits:** checks each feature separately (won't see missing *combinations* like "expensive + mobile"); bin count (10) and row threshold (50) are conventions — tune them in `AuditConfig`.

In [8]:
def check_ranges(df, numeric_cols, pred_ranges, cfg):
    pred_ranges = pred_ranges or {}
    worst, msgs = "PASS", []
    for c in numeric_cols:
        lo, hi = pred_ranges.get(
            c, (df[c].quantile(0.005), df[c].quantile(0.995)))
        edges = np.linspace(lo, hi, 11)               # 10 bins of the PREDICTION range
        counts, _ = np.histogram(df[c], bins=edges)
        bad = counts < cfg.min_rows_per_range_bin
        if bad.any():
            worst = "FAIL"
            gaps = [f"[{edges[i]:.1f},{edges[i+1]:.1f})"
                    for i in range(10) if bad[i]]
            msgs.append(
                f"'{c}': {bad.sum()}/10 bins of the prediction range "
                f"[{lo:.1f},{hi:.1f}] have <{cfg.min_rows_per_range_bin} training "
                f"rows: {', '.join(gaps[:4])}{'...' if len(gaps) > 4 else ''}. "
                f"The model would extrapolate there. Fix: collect data in the "
                f"gaps or restrict predictions to the covered range.")
    if not msgs:
        msgs.append("all numeric prediction ranges supported by training rows.")
    return _res("ranges", worst, f"{len(numeric_cols)} features checked",
                f">={cfg.min_rows_per_range_bin} rows/bin", " | ".join(msgs), None)

In [31]:


def check_learning_curve(df, y, numeric_cols, categorical_cols, cfg, seed=0):
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    import lightgbm as lgb

    X = pd.get_dummies(df[numeric_cols + categorical_cols],
                       columns=categorical_cols, drop_first=True).astype(float)

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    pts = []
    for f in cfg.lc_fractions:
        m = max(int(len(X_tr) * f), 50)
        mdl = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.07,
                                 class_weight="balanced",
                                 random_state=seed, verbose=-1)
        mdl.fit(X_tr.iloc[:m], y_tr.iloc[:m])
        pts.append((m, roc_auc_score(y_te, mdl.predict_proba(X_te)[:, 1])))
    ns = np.array([a for a, _ in pts]); aucs = np.array([b for _, b in pts])
    rising = (aucs[-1] - aucs[-2] > cfg.lc_rise_eps) or \
             (aucs[-1] - aucs[-3] > 2 * cfg.lc_rise_eps)
    curve = ", ".join(f"{a}:{b:.3f}" for a, b in pts)
    note = (" Curve is noisy (few clicks per point); treat numbers as rough."
            if np.any(np.diff(aucs) < -0.02) else "")

    if aucs[-1] >= cfg.target_auc:                    # target already met
        extra = "still rising -- more rows are optional bonus" if rising else "flat"
        return _res("learning_curve", "PASS", f"AUC {aucs[-1]:.3f}",
                    f">={cfg.target_auc}",
                    f"Target reached: AUC {aucs[-1]:.3f} ({extra}) "
                    f"({curve}).{note}", None)

    if rising:                                        # below target, still climbing
        n_req, tail = None, ""
        try:                                          # AUC(n) = A - B * n^(-beta)
            from scipy.optimize import curve_fit
            fun = lambda n, A, B, b: A - B * n**(-b)
            (A, B, b), _ = curve_fit(
                fun, ns, aucs, p0=[aucs[-1] + 0.03, 1.0, 0.5],
                bounds=([aucs[-1], 1e-6, 0.05], [1.0, 1e3, 2.0]), maxfev=20000)
            if A > cfg.target_auc:
                n_fit = int((B / (A - cfg.target_auc))**(1 / b))
                if n_fit > 20 * ns[-1]:               # too far to trust the fit
                    tail = (f" Power-law fit says ~{n_fit:,} rows -- far beyond "
                            f"reliable extrapolation; read as 'a lot more', "
                            f"collect in steps and re-check.")
                else:
                    n_req = n_fit
                    tail = (f" Power-law fit: plateau ~{A:.3f}; reaching AUC "
                            f"{cfg.target_auc} needs ~{n_req:,} training rows.")
            else:
                tail = (f" Power-law fit: plateau ~{A:.3f} < target "
                        f"{cfg.target_auc} => rows alone will not reach it; "
                        f"new features needed too.")
        except Exception:
            tail = " (extrapolation fit failed; collect more and re-check)"
        return _res("learning_curve", "WARN", f"AUC {aucs[-1]:.3f}, rising",
                    f">={cfg.target_auc}",
                    f"Below target and still rising ({curve}). More rows WILL "
                    f"help.{tail}{note}", n_req)

    return _res("learning_curve", "FAIL", f"AUC {aucs[-1]:.3f}, flat",
                f">={cfg.target_auc}",
                f"Curve is FLAT at {aucs[-1]:.3f} < target {cfg.target_auc} "
                f"({curve}). More rows will NOT help: the missing thing is "
                f"FEATURES (new signals), not rows.{note}", None)


#### Audit

In [28]:
def audit(df, click_col, numeric_cols, categorical_cols, pred_ranges=None, cfg=AuditConfig(), run_learning_curve=True):
    y = df[click_col]
    checks = [
        check_precision(y, cfg),
        check_epv(df, y, numeric_cols, categorical_cols, cfg),
        check_levels(df, y, categorical_cols, cfg),
        check_ranges(df, numeric_cols, pred_ranges, cfg),
    ]
    if run_learning_curve:
        checks.append(check_learning_curve(df, y, numeric_cols,
                                           categorical_cols, cfg))
    rep = pd.DataFrame(checks)

    n_reqs = [c["n_req"] for c in checks if c["n_req"]]
    fails = rep[rep.status == "FAIL"]["check"].tolist()
    warns = rep[rep.status == "WARN"]["check"].tolist()
    lc = next((c for c in checks if c["check"] == "learning_curve"), None)
    feature_gap = lc and lc["status"] == "FAIL"

    if fails:
        verdict = f"FAIL -- failed: {', '.join(fails)}"
        if warns:
            verdict += f"; warnings: {', '.join(warns)}"
        verdict += "."
        if feature_gap:
            verdict += (" Main gap is INFORMATION, not size: add features; "
                        "more rows will not reach the target AUC.")
        elif n_reqs and max(n_reqs) > len(df):
            verdict += (f" Estimated required size: ~{max(n_reqs):,} rows "
                        f"(have {len(df):,}).")
    elif warns:
        verdict = (f"PASS WITH WARNINGS ({', '.join(warns)}) -- core "
                   f"requirements met with {len(df):,} rows / "
                   f"{int(y.sum()):,} clicks; see reasons above.")
    else:
        verdict = (f"PASS -- data is sufficient: {len(df):,} rows / "
                   f"{int(y.sum()):,} clicks cover precision, parameters, "
                   f"levels, ranges and power.")
    return rep, (max(n_reqs) if n_reqs else None), verdict

def print_audit(title, df, click_col, num, cat, pred_ranges=None, cfg=AuditConfig(), run_lc=True):
    print("\n" + "=" * 78 + f"\nSCENARIO: {title}\n" + "=" * 78)

    p = df[click_col].mean()
    print(f"rows={len(df):,}  clicks={int(df[click_col].sum()):,}  CTR={p:.4f}")

    rep, n_req, verdict = audit(df, click_col, num, cat, pred_ranges, cfg, run_lc)
    for _, r in rep.iterrows():
        print(f"[{r['status']:4}] {r['check']:<15} measured={r['measured']}  "
              f"required={r['required']}")
        print(f"       {r['reason']}")
    print(f"\nVERDICT: {verdict}")

#### A) healthy

In [29]:
n = 60_000; r = np.random.default_rng(1)
price = r.gamma(4, 25, n); rating = np.clip(r.normal(4, .6, n), 1, 5)
novelty = r.uniform(0, 10, n)
brand = r.choice([f"b{i}" for i in range(8)], n,
                    p=[.25, .2, .15, .12, .1, .08, .06, .04])
device = r.choice(["mobile", "desktop", "tablet"], n, p=[.6, .3, .1])
beff = dict(zip([f"b{i}" for i in range(8)], r.normal(0, .25, 8)))
lg = (-2.2 - 0.01*(price-100) + 0.8*(rating-4) + 0.4 - 0.05*(novelty-5)**2
        + pd.Series(brand).map(beff).to_numpy())
dfA = pd.DataFrame({"f_price": price, "f_rating": rating, "f_novelty": novelty,
                    "brand": brand, "device": device,
                    "click": clicks_from(lg, r)})

In [30]:
print_audit("A. healthy dataset (should PASS everything)",
                dfA, "click", ["f_price", "f_rating", "f_novelty"],
                ["brand", "device"])


SCENARIO: A. healthy dataset (should PASS everything)
rows=60,000  clicks=6,979  CTR=0.1163
[(2250, 0.6435127577958903), (4500, 0.6643464811354098), (9000, 0.6729667022986406), (18000, 0.6844388288357424), (31499, 0.6966854482116821), (45000, 0.7002849981463448)]
+++++++++++++
[PASS] precision       measured=+-2.2%  required=+-10%
       CTR=0.1163 known to +-2.2% (relative). Good enough.
[PASS] EPV             measured=581.6  required=>=20
       6,979 clicks / 12 parameters = 581.6 events per variable. Stable fit.
[PASS] levels          measured=brand:8 lvls; device:3 lvls  required=>=20 clicks/level
       'brand': all 8 levels have >=20 clicks; unseen risk 0.0000. | 'device': all 3 levels have >=20 clicks; unseen risk 0.0000.
[PASS] ranges          measured=3 features checked  required=>=50 rows/bin
       all numeric prediction ranges supported by training rows.
[PASS] learning_curve  measured=AUC 0.700  required=>=0.65
       Target reached: AUC 0.700 (still rising -- more rows 

#### B) too few clicks

In [32]:
n = 4_000; r = np.random.default_rng(2)
price = r.gamma(4, 25, n); rating = np.clip(r.normal(4, .6, n), 1, 5)
quality = r.normal(0, 1, n)
brand = r.choice([f"b{i}" for i in range(5)], n)
device = r.choice(["mobile", "desktop", "tablet"], n, p=[.6, .3, .1])
lg = -4.9 - 0.006*(price-100) + 0.5*(rating-4) + 0.2*quality
dfB = pd.DataFrame({"f_price": price, "f_rating": rating, "f_quality": quality,
                    "brand": brand, "device": device,
                    "click": clicks_from(lg, r)})

In [33]:
print_audit("B. rare clicks, small n (should FAIL precision/EPV/power)",
                dfB, "click", ["f_price", "f_rating", "f_quality"],
                ["brand", "device"])


SCENARIO: B. rare clicks, small n (should FAIL precision/EPV/power)
rows=4,000  clicks=30  CTR=0.0075
[FAIL] precision       measured=+-35.7%  required=+-10%
       CTR=0.0075 known to +-35.7% (relative). Target +-10% needs ~50,838 rows.
[FAIL] EPV             measured=3.3  required=>=20
       30 clicks / 9 parameters = 3.3 events per variable. Need >= 20: ~24,000 rows at this CTR, or reduce parameters (bucket rare category levels).
[FAIL] levels          measured=brand:5 lvls; device:3 lvls  required=>=20 clicks/level
       'brand': 5/5 levels have <20 clicks; unseen-level risk (Good-Turing) = 0.0000. At this CTR even a 100%-share level cannot reach 20 clicks -- the binding problem is total clicks (see precision/EPV), not bucketing. | 'device': 3/3 levels have <20 clicks; unseen-level risk (Good-Turing) = 0.0000. At this CTR even a 100%-share level cannot reach 20 clicks -- the binding problem is total clicks (see precision/EPV), not bucketing.
[FAIL] ranges          measured=3 fea

#### C) long-tail categories

In [34]:
n = 30_000; r = np.random.default_rng(3)
n_brands = 300
shares = 1 / np.arange(1, n_brands + 1)**1.15; shares /= shares.sum()
brand = r.choice([f"b{i}" for i in range(n_brands)], n, p=shares)
price = r.gamma(4, 25, n); rating = np.clip(r.normal(4, .6, n), 1, 5)
beff = dict(zip([f"b{i}" for i in range(n_brands)],
                r.normal(0, .3, n_brands)))
lg = (-2.9 - 0.008*(price-100) + 0.6*(rating-4)
        + pd.Series(brand).map(beff).to_numpy())
dfC = pd.DataFrame({"f_price": price, "f_rating": rating, "brand": brand,
                    "click": clicks_from(lg, r)})

In [35]:
print_audit("C. 300 long-tail brand levels (should FAIL levels & EPV)",
                dfC, "click", ["f_price", "f_rating"], ["brand"])


SCENARIO: C. 300 long-tail brand levels (should FAIL levels & EPV)
rows=30,000  clicks=2,011  CTR=0.0670
[PASS] precision       measured=+-4.2%  required=+-10%
       CTR=0.0670 known to +-4.2% (relative). Good enough.
[FAIL] EPV             measured=6.7  required=>=20
       2,011 clicks / 301 parameters = 6.7 events per variable. Need >= 20: ~89,807 rows at this CTR, or reduce parameters (bucket rare category levels).
[FAIL] levels          measured=brand:300 lvls  required=>=20 clicks/level
       'brand': 284/300 levels have <20 clicks; unseen-level risk (Good-Turing) = 0.0000. Fix: bucket levels with share <0.99% into OTHER (cheap), or collect ~2,237,693 rows to keep the rarest level separate (usually not worth it).
[PASS] ranges          measured=2 features checked  required=>=50 rows/bin
       all numeric prediction ranges supported by training rows.
[WARN] learning_curve  measured=AUC 0.634, rising  required=>=0.65
       Below target and still rising (1125:0.562, 2250:0.566,

#### D) range gap

In [36]:
n = 25_000; r = np.random.default_rng(4)
price = r.uniform(10, 60, n)                       # training only covers 10..60
novelty = np.concatenate([r.uniform(0, 3, n//2),   # internal hole 3..7
                            r.uniform(7, 10, n - n//2)])
rating = np.clip(r.normal(4, .6, n), 1, 5)
device = r.choice(["mobile", "desktop"], n, p=[.6, .4])
lg = -2.3 - 0.02*(price-35) + 0.7*(rating-4)
dfD = pd.DataFrame({"f_price": price, "f_novelty": novelty,
                    "f_rating": rating, "device": device,
                    "click": clicks_from(lg, r)})

In [37]:
print_audit("D. prediction range wider than training (should FAIL ranges)",
                dfD, "click", ["f_price", "f_novelty", "f_rating"], ["device"],
                pred_ranges={"f_price": (10, 200), "f_novelty": (0, 10)})


SCENARIO: D. prediction range wider than training (should FAIL ranges)
rows=25,000  clicks=2,508  CTR=0.1003
[PASS] precision       measured=+-3.7%  required=+-10%
       CTR=0.1003 known to +-3.7% (relative). Good enough.
[PASS] EPV             measured=627.0  required=>=20
       2,508 clicks / 4 parameters = 627.0 events per variable. Stable fit.
[PASS] levels          measured=device:2 lvls  required=>=20 clicks/level
       'device': all 2 levels have >=20 clicks; unseen risk 0.0000.
[FAIL] ranges          measured=3 features checked  required=>=50 rows/bin
       'f_price': 7/10 bins of the prediction range [10.0,200.0] have <50 training rows: [67.0,86.0), [86.0,105.0), [105.0,124.0), [124.0,143.0).... The model would extrapolate there. Fix: collect data in the gaps or restrict predictions to the covered range. | 'f_novelty': 4/10 bins of the prediction range [0.0,10.0] have <50 training rows: [3.0,4.0), [4.0,5.0), [5.0,6.0), [6.0,7.0). The model would extrapolate there. Fix: co

#### E) still rising

In [17]:
n = 2_500; r = np.random.default_rng(5)
price = r.gamma(4, 25, n); rating = np.clip(r.normal(4, .6, n), 1, 5)
novelty = r.uniform(0, 10, n)
delivery = r.integers(1, 10, n).astype(float)
device = r.choice(["mobile", "desktop", "tablet"], n, p=[.6, .3, .1])
lg = (-2.1 - 0.012*(price-100) + 0.9*(rating-4) + 0.5 - 0.055*(novelty-5)**2
        - 0.16*(delivery-5)*(device == "mobile"))
dfE = pd.DataFrame({"f_price": price, "f_rating": rating, "f_novelty": novelty,
                    "f_delivery_days": delivery, "device": device,
                    "click": clicks_from(lg, r)})

In [38]:
print_audit("E. rich signal but only 2,500 rows (learning curve should RISE)",
                dfE, "click",
                ["f_price", "f_rating", "f_novelty", "f_delivery_days"],
                ["device"])


SCENARIO: E. rich signal but only 2,500 rows (learning curve should RISE)
rows=2,500  clicks=344  CTR=0.1376
[PASS] precision       measured=+-9.8%  required=+-10%
       CTR=0.1376 known to +-9.8% (relative). Good enough.
[PASS] EPV             measured=57.3  required=>=20
       344 clicks / 6 parameters = 57.3 events per variable. Stable fit.
[PASS] levels          measured=device:3 lvls  required=>=20 clicks/level
       'device': all 3 levels have >=20 clicks; unseen risk 0.0000.
[FAIL] ranges          measured=4 features checked  required=>=50 rows/bin
       'f_price': 2/10 bins of the prediction range [18.6,268.5] have <50 training rows: [218.5,243.5), [243.5,268.5). The model would extrapolate there. Fix: collect data in the gaps or restrict predictions to the covered range. | 'f_rating': 1/10 bins of the prediction range [2.4,5.0] have <50 training rows: [2.4,2.7). The model would extrapolate there. Fix: collect data in the gaps or restrict predictions to the covered range. 

#### F) weak features

In [39]:
n = 40_000; r = np.random.default_rng(6)
x1, x2, x3 = r.normal(0, 1, n), r.normal(0, 1, n), r.normal(0, 1, n)
device = r.choice(["mobile", "desktop"], n, p=[.5, .5])
lg = -2.45 + 0.06*x1 + 0.05*x2 + 0.04*x3
dfF = pd.DataFrame({"f_x1": x1, "f_x2": x2, "f_x3": x3, "device": device,
                    "click": clicks_from(lg, r)})

In [40]:
print_audit("F. 40k rows but nearly useless features "
                "(curve should be FLAT BELOW target => missing features)",
                dfF, "click", ["f_x1", "f_x2", "f_x3"], ["device"])


SCENARIO: F. 40k rows but nearly useless features (curve should be FLAT BELOW target => missing features)
rows=40,000  clicks=3,248  CTR=0.0812
[PASS] precision       measured=+-3.3%  required=+-10%
       CTR=0.0812 known to +-3.3% (relative). Good enough.
[PASS] EPV             measured=812.0  required=>=20
       3,248 clicks / 4 parameters = 812.0 events per variable. Stable fit.
[PASS] levels          measured=device:2 lvls  required=>=20 clicks/level
       'device': all 2 levels have >=20 clicks; unseen risk 0.0000.
[PASS] ranges          measured=3 features checked  required=>=50 rows/bin
       all numeric prediction ranges supported by training rows.
[FAIL] learning_curve  measured=AUC 0.506, flat  required=>=0.65
       Curve is FLAT at 0.506 < target 0.65 (1500:0.518, 3000:0.528, 6000:0.511, 12000:0.508, 21000:0.510, 30000:0.506). More rows will NOT help: the missing thing is FEATURES (new signals), not rows.

VERDICT: FAIL -- failed: learning_curve. Main gap is INFORMATIO